# Tutorial

This notebook runs the whole pipeline on a cohort it makes up as it goes, so
you can follow it with nothing installed but the package:

```bash
pip install manifold-genetics
```

It takes well under a minute. Every output below is produced when the
documentation is built, so nothing here can drift out of date without the
build failing.

Simulated genotypes are a teaching device, not a validation: for what the
pipeline does on real cohorts, see [Testing against real
cohorts](testing-real-cohorts.md).

In [ ]:
%matplotlib inline

import json
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# tqdm looks for ipywidgets and warns when a rendered page has none.
import warnings

warnings.filterwarnings('ignore', message='IProgress not found')

work = Path(tempfile.mkdtemp(prefix='mg-tutorial-'))
rng = np.random.default_rng(0)
work

## 1. A cohort to work with

Three populations that share an ancestral allele frequency and have drifted
apart from it. This is the Balding-Nichols model: draw an ancestral frequency
`p`, then draw each population's frequency from a Beta distribution centred on
`p` whose spread is set by `f_st`. Larger `f_st` means more differentiated
populations and structure that is easier to see.

Then two groups of admixed individuals -- Alpha with Beta, and Beta with
Gamma -- each person carrying their own mixture proportion. That is closer to
a real cohort, and it matters for the embedding further down.

Nothing about this is specific to the package -- it is just a way to get a
genotype matrix with known structure in it.

In [ ]:
N_PER_POPULATION = 100
N_ADMIXED = 60
N_VARIANTS = 3000
F_ST = 0.03  # roughly the differentiation between continental human groups
POPULATIONS = ['Alpha', 'Beta', 'Gamma']

ancestral = rng.uniform(0.1, 0.9, size=N_VARIANTS)
scale = (1 - F_ST) / F_ST
frequencies = np.stack(
    [rng.beta(ancestral * scale, (1 - ancestral) * scale) for _ in POPULATIONS]
)

# Genotypes are the count of the A1 allele: 0, 1 or 2.
blocks = [
    rng.binomial(2, frequencies[i], size=(N_PER_POPULATION, N_VARIANTS))
    for i in range(len(POPULATIONS))
]
labels_by_block = [[name] * N_PER_POPULATION for name in POPULATIONS]

# Two groups of admixed individuals, each person with their own mixture
# proportion. Real cohorts are continuous like this, and it matters here:
# three cleanly separated clusters give PHATE a disconnected neighbour graph,
# which it will (rightly) warn about and cannot lay out sensibly.
for left, right in [(0, 1), (1, 2)]:
    mixture = rng.uniform(0.15, 0.85, size=N_ADMIXED)[:, None]
    blocks.append(
        rng.binomial(
            2, mixture * frequencies[left] + (1 - mixture) * frequencies[right]
        )
    )
    name = f'{POPULATIONS[left]}-{POPULATIONS[right]} admixed'
    labels_by_block.append([name] * N_ADMIXED)

dosages = np.concatenate(blocks)
population = np.array([name for block in labels_by_block for name in block])

sample_ids = [f'S{i:04d}' for i in range(len(dosages))]
dosages.shape

### Writing it as PLINK files

The pipeline reads PLINK 1 binary genotypes, so the cohort has to be written as
a `.bed`/`.bim`/`.fam` triple. The `.bed` format is a three-byte magic header
followed, for each variant, by two bits per sample -- so four samples per byte:

| code | meaning | dosage (count of A1) |
|---|---|---|
| `00` | homozygous A1 | 2 |
| `01` | missing | -- |
| `10` | heterozygous | 1 |
| `11` | homozygous A2 | 0 |

Note that the codes do not run in dosage order, and that A1 is the *fifth*
column of the `.bim`. Getting either backwards produces a PCA that looks
plausible and is wrong, which is why the package determined this convention
empirically against reference outputs rather than from documentation.

In [ ]:
DOSAGE_TO_CODE = {2: 0b00, 1: 0b10, 0: 0b11}


def write_plink(prefix, dosages, sample_ids):
    """Write a dosage matrix (samples x variants) as a PLINK 1 binary triple."""
    prefix = Path(prefix)
    n_samples, n_variants = dosages.shape

    codes = np.vectorize(DOSAGE_TO_CODE.get)(dosages.T).astype(np.uint8)
    # Pad to a multiple of four samples; the spare bits are ignored on read.
    padding = (-n_samples) % 4
    if padding:
        codes = np.pad(codes, ((0, 0), (0, padding)))
    shifts = 2 * (np.arange(codes.shape[1]) % 4)
    packed = np.bitwise_or.reduce(
        (codes << shifts).reshape(n_variants, -1, 4), axis=2
    ).astype(np.uint8)

    with open(prefix.with_suffix('.bed'), 'wb') as bed:
        bed.write(bytes([0x6C, 0x1B, 0x01]))
        bed.write(packed.tobytes())

    with open(prefix.with_suffix('.bim'), 'w') as bim:
        for i in range(n_variants):
            # chromosome, id, centimorgans, position, A1, A2
            bim.write(f'1\trs{i}\t0\t{i + 1}\tA\tG\n')

    with open(prefix.with_suffix('.fam'), 'w') as fam:
        for sample in sample_ids:
            # family, individual, father, mother, sex, phenotype
            fam.write(f'{sample}\t{sample}\t0\t0\t0\t-9\n')

    return prefix


write_plink(work / 'cohort', dosages, sample_ids)
sorted(p.name for p in work.glob('cohort.*'))

## 2. PCA

`PCA` needs no external binary: it reads the `.bed` directly and computes a
randomized SVD in process. It reproduces FlashPCA's conventions -- dosage as
the count of A1, `binom2` standardisation, eigenvalues as `S**2/n_variants` --
so a model fitted here can be read by FlashPCA and the other way round.

In [ ]:
from manifold_genetics import PCA

pca = PCA(n_components=10)
coords = pca.fit_transform(work / 'cohort', output_path=work / 'pca.csv')
coords.head()

The output is the format every step of this package speaks: a `sample_id`
column followed by `dim_1 ... dim_n`. Embeddings, metrics and plots all read
and write it, so the steps compose in any order that makes sense.

The first two components should separate three populations that drifted apart
independently:

In [ ]:
labels = pd.DataFrame({'sample_id': sample_ids, 'population': population})
merged = coords.merge(labels, on='sample_id')

fig, ax = plt.subplots(figsize=(5, 4.5))
for name, group in merged.groupby('population'):
    ax.scatter(group['dim_1'], group['dim_2'], s=14, label=name, alpha=0.8)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.legend(title='population', frameon=False)
ax.set_title('PCA of the simulated cohort')
fig.tight_layout()
fig

How much of the variance in the first ten components does population
membership account for? Compared against shuffled labels, so the number means
the same thing whatever the cohort size or the number of groups:

In [ ]:
def variance_explained(X, y):
    grand = X.mean(axis=0)
    total = ((X - grand) ** 2).sum()
    between = sum(
        (y == g).sum() * ((X[y == g].mean(axis=0) - grand) ** 2).sum()
        for g in np.unique(y)
    )
    return between / total


X = merged[[f'dim_{i}' for i in range(1, 11)]].to_numpy()
y = merged['population'].to_numpy()

observed = variance_explained(X, y)
shuffled = np.mean([variance_explained(X, rng.permutation(y)) for _ in range(5)])

print(f'population explains {observed:.1%} of the variance in PC1-10')
print(f'shuffled labels explain {shuffled:.1%}')
print(f'ratio: {observed / shuffled:.0f}x')

## 3. A manifold embedding

PHATE, UMAP, t-SNE and diffusion maps all have the same three methods --
`fit`, `transform`, `fit_transform` -- and all read the CSV format above, so
they are interchangeable.

Embeddings run on the principal components rather than on raw genotypes: the
PCs are where the population structure is, and the rest is mostly noise that
would swamp the neighbour graph.

What to look for: the admixed individuals should lie along paths between the
populations they are mixtures of, rather than in clusters of their own, giving
a connected structure with Beta in the middle. Recovering continuous structure
like that is the reason to reach for PHATE over a scatter of PC1 against PC2.

In [ ]:
from manifold_genetics import PHATE

# knn=30 suits 300 samples. On a cohort of tens of thousands, use the
# `subsample` preset's settings instead -- knn=500 with 10,000 random
# landmarks. See Concepts for why.
embedding = PHATE(knn=30, t=3).fit_transform(work / 'pca.csv')
embedding.head()

In [ ]:
merged_embedding = embedding.merge(labels, on='sample_id')

fig, ax = plt.subplots(figsize=(5, 4.5))
for name, group in merged_embedding.groupby('population'):
    ax.scatter(group['dim_1'], group['dim_2'], s=14, label=name, alpha=0.8)
ax.set_xlabel('PHATE 1')
ax.set_ylabel('PHATE 2')
ax.legend(title='population', frameon=False)
ax.set_title('PHATE embedding of the simulated cohort')
fig.tight_layout()
fig

## 4. The whole thing from a config file

Running the steps by hand is useful for understanding them. For real work the
entry point is a config file, which records every setting in one reviewable
place -- the reason the examples in this repository stopped being shell
scripts.

A config needs labels and a colormap. The colormap is keyed by label column,
then by value, so one file can colour several groupings of the same cohort:

In [ ]:
labels.to_csv(work / 'labels.csv', index=False)

colormap = {
    'population': {
        'Alpha': '#4C72B0',
        'Beta': '#DD8452',
        'Gamma': '#55A868',
        'Alpha-Beta admixed': '#C44E52',
        'Beta-Gamma admixed': '#8172B3',
    }
}
(work / 'colors.json').write_text(json.dumps(colormap, indent=2))

# The fit set is the subset PCA is fitted on; the project set is what gets
# projected onto it. Here the project set contains the fit set, which is the
# `whole_cohort` mode.
fit = np.arange(len(dosages)) % 3 != 0
write_plink(work / 'fit', dosages[fit], [s for s, keep in zip(sample_ids, fit) if keep])

config = f'''
preset: whole_cohort

data:
  fit_plink: fit
  project_plink: cohort
  labels: labels.csv
  colormap: colors.json
  output_dir: outputs

pca:
  n_pcs: 10

embedding:
  method: phate
  knn: 30

visualization:
  admix_group_column: population

skip:
  admixture: true
'''
(work / 'config.yaml').write_text(config)
print(config)

Before running anything, check what the config actually resolves to. `--dry-run`
prints the full call, with paths resolved against the config file and
`(default)` marking every setting the package supplied rather than the file:

In [ ]:
!manifold-genetics run {work}/config.yaml --dry-run

`embedding_input: project` came from the preset, not the file. That is the
kind of setting worth looking at before committing to a run that takes hours.

Now run it. The same thing happens from the command line with
`manifold-genetics run config.yaml`:

In [ ]:
from manifold_genetics.pipeline.configfile import load_config
from manifold_genetics.pipeline.runner import run_pipeline

result = run_pipeline(**load_config(work / 'config.yaml'))

for path in sorted((work / 'outputs').rglob('*')):
    if path.is_file():
        print(path.relative_to(work / 'outputs'))

The output tree is the same shape for every cohort, which is what makes
results from different runs comparable. `result` is a typed object rather than
a dictionary of paths:

In [ ]:
print('PCA      ', result.pca.project_pca.name)
print('embedding', result.embedding.embedding_file.name)
print('figures  ', [p.name for p in result.figures])
print('failed   ', result.failed_steps)

In [ ]:
from IPython.display import Image

Image(filename=str(result.embedding_figures[0]))

## Where to go next

- [Concepts](concepts.md) -- fit versus project, the three modes, and why
  landmarking settings matter above about fifty thousand samples.
- [Configuration](configuration.md) -- every key a config file accepts.
- [Running on a cluster](hpc.md) -- SLURM, and the memory arithmetic for
  cohorts that do not fit in RAM.
- [Controlled-access data](controlled-access.md) -- UK Biobank and All of Us.

`manifold-genetics setup` downloads plink2 and plink, which the example
`prepare_data.sh` scripts use to build fit and project subsets from a full
cohort. Nothing in this notebook needed them.